# 🌿 01 Preprocessing — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/01_preprocessing.ipynb`           |
| **Tujuan** | Membersihkan data sensor, normalisasi Min-Max, bentuk sekuens waktu, labeling klasifikasi penyiraman, split train/test, simpan artefak. |
| **Input**  | `data/raw/dataset_bonsai.csv`               |
| **Output** | `artifacts/data_train.npz`, `artifacts/data_test.npz`, `artifacts/scaler_bonsai.pkl`, `artifacts/label_info.json` |
| **Urutan** | Jalankan pertama (tidak ada notebook sebelumnya) |

In [20]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : d:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [ ]:
# ── IMPORT LIBRARY & KONSTANTA GLOBAL ──────────────────────────────────
import random, os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import joblib, json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

# Verifikasi versi library
import sklearn, matplotlib
try:
    import importlib.metadata
    seaborn_ver = importlib.metadata.version("seaborn")
except:
    seaborn_ver = "unknown"
    
libs = {
    "TensorFlow"  : tf.__version__,
    "Scikit-learn": sklearn.__version__,
    "Pandas"      : pd.__version__,
    "NumPy"       : np.__version__,
    "Matplotlib"  : matplotlib.__version__,
    "Seaborn"     : seaborn_ver,
}
for lib, ver in libs.items():
    print(f"  {lib:<14}: {ver}")

# Konstanta Preprocessing
DATA_PATH      = "../data/raw/dataset_bonsai.csv"
ARTIFACTS_DIR  = "../artifacts"
TRAIN_RATIO    = 0.80
LOOK_BACK      = 24
SOIL_THRESHOLD = 60.0
FEATURES       = ["temperature_c", "humidity_air_pct", "soil_moisture_pct"]

BOUNDS = {
    "temperature_c"     : (10.0, 45.0),
    "humidity_air_pct"  : (0.0,  100.0),
    "soil_moisture_pct" : (0.0,  100.0),
}
STAGNANT_WINDOW = 12

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("\n✅ Konfigurasi siap.")

[SEED] Global random seed = 42
  TensorFlow    : 2.21.0
  Scikit-learn  : 1.4.0
  Pandas        : 2.2.0
  NumPy         : 1.26.3
  Matplotlib    : 3.8.2
  Seaborn       : 0.13.2

✅ Konfigurasi siap.


In [22]:
# ── RULE-CLEAN-01: Parsing & Pengurutan Timestamp ──────────────────────
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"[CLEAN-01] Timestamp diparse. Rentang: {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
print(f"[CLEAN-01] Total baris awal: {len(df)}")

[CLEAN-01] Timestamp diparse. Rentang: 2025-01-01 00:00:00 → 2025-03-31 23:30:00
[CLEAN-01] Total baris awal: 4320


In [23]:
# ── RULE-CLEAN-02: Hapus Nilai Kosong (Missing Values) ──────────────────
n_null = df.isnull().sum().sum()
df = df.dropna().reset_index(drop=True)
print(f"[CLEAN-02] Nilai null dihapus: {n_null} | Sisa baris: {len(df)}")

[CLEAN-02] Nilai null dihapus: 0 | Sisa baris: 4320


In [24]:
# ── RULE-CLEAN-03: Hapus Outlier Nilai Ekstrem ─────────────────────────
mask_valid = pd.Series([True] * len(df))
for col, (lo, hi) in BOUNDS.items():
    invalid = ~df[col].between(lo, hi, inclusive="both")
    print(f"[CLEAN-03] {col}: {invalid.sum()} outlier ditemukan")
    mask_valid &= ~invalid
df = df[mask_valid].reset_index(drop=True)
print(f"[CLEAN-03] Sisa setelah hapus outlier: {len(df)}")

[CLEAN-03] temperature_c: 0 outlier ditemukan
[CLEAN-03] humidity_air_pct: 0 outlier ditemukan
[CLEAN-03] soil_moisture_pct: 0 outlier ditemukan
[CLEAN-03] Sisa setelah hapus outlier: 4320


In [25]:
# ── RULE-CLEAN-04: Hapus Segmen Stagnan (Flat Signal) ───────────────────
stagnant_mask = pd.Series([False] * len(df))
for col in FEATURES:
    rolling_std = df[col].rolling(window=STAGNANT_WINDOW).std()
    stagnant_mask |= (rolling_std == 0)
n_stagnant = stagnant_mask.sum()
df = df[~stagnant_mask].reset_index(drop=True)
print(f"[CLEAN-04] Titik stagnan dihapus: {n_stagnant} | Sisa: {len(df)}")

[CLEAN-04] Titik stagnan dihapus: 0 | Sisa: 4320


In [26]:
# ── RULE-CLEAN-05: Hapus Duplikat Timestamp ─────────────────────────────
n_dup = df.duplicated(subset="timestamp").sum()
df = df.drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)
print(f"[CLEAN-05] Duplikat timestamp dihapus: {n_dup} | Sisa: {len(df)}")

[CLEAN-05] Duplikat timestamp dihapus: 0 | Sisa: 4320


In [27]:
# ── RULE-PRE-03: Split Data Training / Testing (Kronologis) ────────────
split_idx = int(len(df) * TRAIN_RATIO)
df_train  = df.iloc[:split_idx].reset_index(drop=True)
df_test   = df.iloc[split_idx:].reset_index(drop=True)

print(f"[SPLIT] Training : {len(df_train)} baris "
      f"({df_train['timestamp'].iloc[0]} → {df_train['timestamp'].iloc[-1]})")
print(f"[SPLIT] Testing  : {len(df_test)} baris "
      f"({df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]})")

[SPLIT] Training : 3456 baris (2025-01-01 00:00:00 → 2025-03-13 23:30:00)
[SPLIT] Testing  : 864 baris (2025-03-14 00:00:00 → 2025-03-31 23:30:00)


In [28]:
# ── RULE-PRE-04: Normalisasi Min-Max Scaling ────────────────────────────
scaler       = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(df_train[FEATURES])
test_scaled  = scaler.transform(df_test[FEATURES])

joblib.dump(scaler, f"{ARTIFACTS_DIR}/scaler_bonsai.pkl")
print(f"[NORM] Scaler disimpan → artifacts/scaler_bonsai.pkl")
print(f"[NORM] Min per fitur   : {dict(zip(FEATURES, scaler.data_min_))}")
print(f"[NORM] Max per fitur   : {dict(zip(FEATURES, scaler.data_max_))}")

[NORM] Scaler disimpan → artifacts/scaler_bonsai.pkl
[NORM] Min per fitur   : {'temperature_c': 15.51, 'humidity_air_pct': 52.09, 'soil_moisture_pct': 52.0}
[NORM] Max per fitur   : {'temperature_c': 32.46, 'humidity_air_pct': 95.0, 'soil_moisture_pct': 90.0}


In [29]:
# ── RULE-PRE-05: Pembentukan Sekuens Waktu ─────────────────────────────
def create_sequences(data: np.ndarray, look_back: int):
    """
    Membentuk pasangan input-output untuk LSTM.
    X[i] = data[i : i+look_back, :]     shape → (look_back, n_features)
    y[i] = data[i+look_back, 2]         indeks 2 = soil_moisture_pct
    """
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back : i, :])
        y.append(data[i, 2])
    return np.array(X), np.array(y)

X_train, y_train_reg = create_sequences(train_scaled, LOOK_BACK)
X_test,  y_test_reg  = create_sequences(test_scaled,  LOOK_BACK)

print(f"[SEQ] X_train shape : {X_train.shape}  → (sampel, look_back, fitur)")
print(f"[SEQ] X_test  shape : {X_test.shape}")

[SEQ] X_train shape : (3432, 24, 3)  → (sampel, look_back, fitur)
[SEQ] X_test  shape : (840, 24, 3)


In [30]:
# ── RULE-PRE-06: Pelabelan Klasifikasi Penyiraman ───────────────────────
def inverse_soil(y_norm: np.ndarray, scaler, n_features: int) -> np.ndarray:
    """Kembalikan nilai normalized soil ke skala asli (%)."""
    dummy = np.zeros((len(y_norm), n_features))
    dummy[:, 2] = y_norm
    return scaler.inverse_transform(dummy)[:, 2]

def create_labels(soil_values: np.ndarray, threshold: float) -> np.ndarray:
    """Label 1 = butuh siram, 0 = tidak butuh siram."""
    return (soil_values < threshold).astype(int)

y_train_orig = inverse_soil(y_train_reg, scaler, len(FEATURES))
y_test_orig  = inverse_soil(y_test_reg,  scaler, len(FEATURES))
y_train_cls  = create_labels(y_train_orig, SOIL_THRESHOLD)
y_test_cls   = create_labels(y_test_orig,  SOIL_THRESHOLD)

print(f"[LABEL] TRAIN → Tidak Siram (0): {(y_train_cls==0).sum()} | Siram (1): {(y_train_cls==1).sum()}")
print(f"[LABEL] TEST  → Tidak Siram (0): {(y_test_cls==0).sum()}  | Siram (1): {(y_test_cls==1).sum()}")

rasio = max((y_train_cls==0).sum(), (y_train_cls==1).sum()) / \
        max(min((y_train_cls==0).sum(), (y_train_cls==1).sum()), 1)
print(f"[LABEL] Rasio kelas (mayor:minor) = {rasio:.2f}x {'⚠️ Imbalanced' if rasio > 4 else '✅ Balanced'}")

[LABEL] TRAIN → Tidak Siram (0): 3378 | Siram (1): 54
[LABEL] TEST  → Tidak Siram (0): 574  | Siram (1): 266
[LABEL] Rasio kelas (mayor:minor) = 62.56x ⚠️ Imbalanced


In [31]:
# ── RULE-PRE-07: Simpan Semua Artefak Preprocessing ────────────────────
np.savez_compressed(
    f"{ARTIFACTS_DIR}/data_train.npz",
    X_train=X_train, y_train_cls=y_train_cls, y_train_reg=y_train_orig
)
np.savez_compressed(
    f"{ARTIFACTS_DIR}/data_test.npz",
    X_test=X_test, y_test_cls=y_test_cls, y_test_reg=y_test_orig
)

label_info = {
    "soil_threshold" : SOIL_THRESHOLD,
    "look_back"      : LOOK_BACK,
    "train_ratio"    : TRAIN_RATIO,
    "features"       : FEATURES,
    "n_features"     : len(FEATURES),
    "train_label_0"  : int((y_train_cls == 0).sum()),
    "train_label_1"  : int((y_train_cls == 1).sum()),
    "test_label_0"   : int((y_test_cls  == 0).sum()),
    "test_label_1"   : int((y_test_cls  == 1).sum()),
}
with open(f"{ARTIFACTS_DIR}/label_info.json", "w") as f:
    json.dump(label_info, f, indent=2)

print("[SAVE] ✅ artifacts/data_train.npz")
print("[SAVE] ✅ artifacts/data_test.npz")
print("[SAVE] ✅ artifacts/scaler_bonsai.pkl")
print("[SAVE] ✅ artifacts/label_info.json")

[SAVE] ✅ artifacts/data_train.npz
[SAVE] ✅ artifacts/data_test.npz
[SAVE] ✅ artifacts/scaler_bonsai.pkl
[SAVE] ✅ artifacts/label_info.json


In [32]:
# ── RULE-ROOF-01: Fungsi Kontrol Atap (Rule-based) ──────────────────────
def roof_control(temp_c: float, humidity_air: float, soil_moisture: float) -> str:
    """
    Menentukan status atap berdasarkan nilai sensor saat ini.

    TUTUP atap jika salah satu kondisi terpenuhi:
      - humidity_air   > 85%   → potensi hujan / kelembapan ekstrem
      - soil_moisture  > 85%   → tanah sudah sangat lembab
      - temp_c         < 15°C  → suhu terlalu rendah (risiko frost)

    Sensor error (nilai di luar batas valid):
      → default TUTUP (keamanan lebih utama dari optimasi)

    Returns: "TUTUP" | "BUKA"
    """
    sensor_ok = (
        10 <= temp_c <= 45 and
        0  <  humidity_air  <= 100 and
        0  <  soil_moisture <= 100
    )
    if not sensor_ok:
        return "TUTUP"

    if humidity_air > 85 or soil_moisture > 85 or temp_c < 15:
        return "TUTUP"
    return "BUKA"

print("✅ Fungsi roof_control siap.")

✅ Fungsi roof_control siap.


## 📊 Ringkasan Preprocessing

**Data yang dihasilkan:**
- `data_train.npz` → X_train, y_train_cls, y_train_reg
- `data_test.npz`  → X_test, y_test_cls, y_test_reg
- `scaler_bonsai.pkl` → MinMaxScaler
- `label_info.json` → Metadata & threshold

**Langkah selanjutnya:**
Jalankan `notebooks/02_training.ipynb` untuk membangun dan melatih model LSTM.